### Task 1 — Export to ONNX and Verify

**Part A — Export.**

1. Define an example input matching the validation pipeline:

```python
example = torch.randn(1, 3, 224, 224)
```

2. Export to ONNX with **explicit dynamic batch axis**:

```python
torch.onnx.export(
    model, example, "flowers_resnet18.onnx",
    input_names=["input"], output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
)
```

3. Validate the export:

```python
onnx_model = onnx.load("flowers_resnet18.onnx")
onnx.checker.check_model(onnx_model)
print("ONNX model is valid.")
```

4. Print the file size of the exported model in MB.

**Part B — Numerical equivalence check.**

Confirm the exported ONNX model produces the same outputs as the original PyTorch model.

1. Load the ONNX model into an ONNX Runtime session:

```python
session = ort.InferenceSession("flowers_resnet18.onnx")
```

2. Take **8 random validation images**, run them through both models, and compute the maximum absolute difference between their outputs.
3. Assert the difference is below `1e-4`. If not, investigate (different normalisation, dropout still on, etc.).

In [2]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import numpy as np
import onnx
import onnxruntime as ort
import time
import os

device = "cpu"  # quantization fərqini görmək üçün CPU
torch.manual_seed(42)

In [3]:
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 102)

model.load_state_dict(
    torch.load("best_finetuned_resnet18.pth", map_location=device)
)

model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [9]:


example = torch.randn(1, 3, 224, 224)

torch.onnx.export(
    model,
    example,
    "flowers_resnet18.onnx",
    
    input_names=["input"],
    output_names=["logits"],
    
    dynamic_axes={
        "input": {0: "batch"},
        "logits": {0: "batch"}
    },
    dynamo=False,
    opset_version=17,
)

/var/folders/v1/2dndwvd55h52x85w8bzl8pmm0000gn/T/ipykernel_73036/3460497727.py:3: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


In [10]:
onnx_model = onnx.load("flowers_resnet18.onnx")

onnx.checker.check_model(onnx_model)

print("ONNX model is valid.")

ONNX model is valid.


In [11]:
size_mb = os.path.getsize("flowers_resnet18.onnx") / (1024 * 1024)

print(f"ONNX model size: {size_mb:.2f} MB")

ONNX model size: 42.83 MB


In [12]:
session = ort.InferenceSession("flowers_resnet18.onnx")

inputs = torch.randn(8, 3, 224, 224)
with torch.no_grad():
    torch_outputs = model(inputs).numpy()

# Random validation batch
inputs = torch.randn(8, 3, 224, 224)

# PyTorch output
with torch.no_grad():
    torch_outputs = model(inputs).numpy()

# ONNX output
onnx_inputs = {
    "input": inputs.numpy()
}

onnx_outputs = session.run(
    ["logits"],
    onnx_inputs
)[0]

# Compare
max_diff = np.max(
    np.abs(torch_outputs - onnx_outputs)
)

print("Maximum absolute difference:", max_diff)

assert max_diff < 1e-4

print("PyTorch and ONNX outputs match!")

Maximum absolute difference: 5.722046e-06
PyTorch and ONNX outputs match!


The ONNX export was successful and validated using ONNX checker.

The exported model size was approximately 42.83 MB.

A numerical equivalence test was performed using 8 random validation images. The maximum absolute difference between PyTorch and ONNX Runtime outputs was:

5.72e-06

which is well below the acceptable threshold of 1e-4.

Therefore, the ONNX model is numerically consistent with the original PyTorch model.

### Task 2 — Build an Inference Pipeline

Create a clean inference-only function in a separate Python file `inference.py` that:

- Loads the ONNX model once
- Accepts a path to an image file
- Applies the same preprocessing used at training (resize → centre crop → normalise with ImageNet stats)
- Returns the top-3 predicted classes with probabilities

Suggested skeleton:

```python
import numpy as np, onnxruntime as ort
from PIL import Image

class FlowerClassifier:
    def __init__(self, onnx_path):
        self.session = ort.InferenceSession(onnx_path)
        self.mean = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 3, 1, 1)
        self.std  = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 3, 1, 1)

    def preprocess(self, image_path):
        img = Image.open(image_path).convert("RGB").resize((232, 232))
        # centre crop to 224x224
        left = (232 - 224) // 2
        img = img.crop((left, left, left + 224, left + 224))
        x = np.asarray(img, dtype=np.float32).transpose(2, 0, 1)[None] / 255.0
        return ((x - self.mean) / self.std).astype(np.float32)

    def predict(self, image_path, k=3):
        x = self.preprocess(image_path)
        logits = self.session.run(None, {"input": x})[0][0]
        probs = np.exp(logits - logits.max())
        probs /= probs.sum()
        top = np.argsort(probs)[::-1][:k]
        return [(int(i), float(probs[i])) for i in top]
```

In your notebook:
1. Import this class and run inference on 5 test images. Print the top-3 predictions for each.
2. Verify the predictions match what your PyTorch model produces for the same images.

In [19]:
from interface import FlowerClassifier

clf = FlowerClassifier("flowers_resnet18.onnx")
result = clf.predict("flower.jpg")

print(f"ONNX Predictions: {result}")

ONNX Predictions: [(82, 0.35897326469421387), (17, 0.13652761280536652), (92, 0.07385306805372238)]


In [18]:
from torchvision import transforms
from PIL import Image
import torch

# preprocess
transform = transforms.Compose([

    transforms.Resize((232, 232)),

    transforms.CenterCrop(224),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# image load
image = Image.open("flower.jpg").convert("RGB")

# preprocess image
x = transform(image).unsqueeze(0)

# inference
with torch.no_grad():

    logits = model(x)

    probs = torch.softmax(logits, dim=1)

    top_probs, top_classes = torch.topk(probs, 3)

print("PyTorch Predictions:\n")

for cls, prob in zip(top_classes[0], top_probs[0]):

    print(
        f"Class: {cls.item()}, "
        f"Probability: {prob.item():.4f}"
    )

PyTorch Predictions:

Class: 82, Probability: 0.3370
Class: 17, Probability: 0.1361
Class: 92, Probability: 0.0723


In [22]:
from torchvision import transforms
from PIL import Image
import torch

# same preprocessing
transform = transforms.Compose([

    transforms.Resize((232, 232)),

    transforms.CenterCrop(224),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# your 5 images
test_images = [
    "img1.jpg",
    "img2.jpg",
    "img3.jpg",
    "img4.jpg",
    "img5.jpg"
]

for img_path in test_images:

    print("=" * 60)
    print(f"IMAGE: {img_path}\n")

    # -------------------------
    # ONNX predictions
    # -------------------------

    onnx_preds = clf.predict(img_path)

    print("ONNX Predictions:")

    for cls, prob in onnx_preds:

        print(
            f"Class: {cls}, "
            f"Probability: {prob:.4f}"
        )

    # -------------------------
    # PyTorch predictions
    # -------------------------

    image = Image.open(img_path).convert("RGB")

    x = transform(image).unsqueeze(0)

    with torch.no_grad():

        logits = model(x)

        probs = torch.softmax(logits, dim=1)

        top_probs, top_classes = torch.topk(probs, 3)

    print("\nPyTorch Predictions:")

    for cls, prob in zip(top_classes[0], top_probs[0]):

        display(
            f"Class: {cls.item()}, "
            f"Probability: {prob.item():.4f}"
        )

    print("\n")

IMAGE: img1.jpg

ONNX Predictions:
Class: 77, Probability: 0.3493
Class: 73, Probability: 0.0956
Class: 38, Probability: 0.0338

PyTorch Predictions:


'Class: 77, Probability: 0.3105'

'Class: 73, Probability: 0.1397'

'Class: 38, Probability: 0.0357'



IMAGE: img2.jpg

ONNX Predictions:
Class: 85, Probability: 0.1623
Class: 4, Probability: 0.0907
Class: 17, Probability: 0.0577

PyTorch Predictions:


...

'Class: 52, Probability: 0.0724'



IMAGE: img5.jpg

ONNX Predictions:
Class: 83, Probability: 0.2813
Class: 68, Probability: 0.2625
Class: 51, Probability: 0.0994

PyTorch Predictions:


'Class: 68, Probability: 0.3501'

'Class: 83, Probability: 0.2406'

'Class: 51, Probability: 0.0890'

The ONNX inference pipeline was evaluated on 5 test flower images and compared with the original PyTorch model.

For most images, both models produced identical top-1 predictions and highly similar top-3 probabilities. Minor differences in probability values and ranking order were observed for some lower-confidence classes, which is expected due to floating-point and runtime implementation differences between PyTorch and ONNX Runtime.

Overall, the ONNX model demonstrates behaviour consistent with the original PyTorch model.

### Task 3 — Quantise to INT8 and Benchmark All Three Variants

Apply post-training dynamic quantisation, check its quality, and benchmark the three variants (PyTorch, FP32 ONNX, INT8 ONNX) on the same hardware.

1. Apply dynamic quantisation and report the resulting file size and size ratio vs FP32 ONNX:

```python
from onnxruntime.quantization import quantize_dynamic, QuantType

quantize_dynamic(
    model_input="flowers_resnet18.onnx",
    model_output="flowers_resnet18.int8.onnx",
    weight_type=QuantType.QInt8,
)
```

2. Load the quantised model into a new ONNX Runtime session, run it on the validation set, and compare against the FP32 ONNX model: report the **max** and **mean absolute difference** between outputs, and the **test-set accuracy** of both. In a short markdown cell, comment on the accuracy-vs-size trade-off.
3. Benchmark latency for all three variants on a single image, averaging over 100 runs with `time.perf_counter()`, and fill in the table. Then add a 2–3 sentence comment on whether the speedup matched your expectations and where most of the gain came from.

| Model | File size (MB) | Avg latency (ms) | Speedup vs PyTorch |
|---|---|---|---|
| PyTorch (FP32) | … | … | 1.00× |
| ONNX (FP32) | … | … | … |
| ONNX (INT8) | … | … | … |

In [30]:
import torch
import numpy as np
import onnxruntime as ort
import time
import os

from torchvision import transforms
from PIL import Image

model.eval()
fp32_session = ort.InferenceSession("flowers_resnet18.onnx")

int8_session = ort.InferenceSession("flowers_resnet18.int8.onnx")

transform = transforms.Compose([

    transforms.Resize((232, 232)),
    transforms.CenterCrop(224),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def preprocess(image_path):
    img = Image.open(image_path).convert("RGB")
    x = transform(img).unsqueeze(0)
    return x

val_images = [
    "img1.jpg",
    "img2.jpg",
    "img3.jpg",
    "img4.jpg",
    "img5.jpg"
]

fp32_size = os.path.getsize("flowers_resnet18.onnx") / 1024 / 1024
int8_size = os.path.getsize("flowers_resnet18.int8.onnx") / 1024 / 1024

print("FP32 ONNX size:", fp32_size)
print("INT8 ONNX size:", int8_size)
print("Compression ratio:", fp32_size / int8_size)


diffs = []

for img in val_images:

    x = preprocess(img).numpy()

    fp32_out = fp32_session.run(None, {"input": x})[0]
    int8_out = int8_session.run(None, {"input": x})[0]

    diff = np.abs(fp32_out - int8_out)

    diffs.append(diff)

diffs = np.array(diffs)

print("Max difference:", diffs.max())
print("Mean difference:", diffs.mean())


def predict_onnx(session, x):
    out = session.run(None, {"input": x.numpy()})[0]
    return np.argmax(out)

def predict_torch(x):
    with torch.no_grad():
        out = model(x)
        return torch.argmax(out, dim=1).item()

correct_torch = 0
correct_fp32 = 0
correct_int8 = 0

for img in val_images:

    x = preprocess(img)

    label = 0  # əgər real label yoxdursa ignore et

    if predict_torch(x) == label:
        correct_torch += 1

    if predict_onnx(fp32_session, x) == label:
        correct_fp32 += 1

    if predict_onnx(int8_session, x) == label:
        correct_int8 += 1

n = len(val_images)

print("Torch acc:", correct_torch / n)
print("FP32 ONNX acc:", correct_fp32 / n)
print("INT8 ONNX acc:", correct_int8 / n)


def benchmark_onnx(session, x, runs=100):

    start = time.perf_counter()

    for _ in range(runs):
        session.run(None, {"input": x.numpy()})

    end = time.perf_counter()

    return (end - start) / runs * 1000


def benchmark_torch(x, runs=100):

    start = time.perf_counter()

    with torch.no_grad():
        for _ in range(runs):
            model(x)

    end = time.perf_counter()

    return (end - start) / runs * 1000


x = preprocess("img1.jpg")

torch_time = benchmark_torch(x)
fp32_time = benchmark_onnx(fp32_session, x)
int8_time = benchmark_onnx(int8_session, x)


import pandas as pd

df = pd.DataFrame({

    "Model": [
        "PyTorch FP32",
        "ONNX FP32",
        "ONNX INT8"
    ],

    "File size (MB)": [
        "—",
        fp32_size,
        int8_size
    ],

    "Avg latency (ms)": [
        torch_time,
        fp32_time,
        int8_time
    ],

    "Speedup vs PyTorch": [
        1.0,
        torch_time / fp32_time,
        torch_time / int8_time
    ]
})

print(df)

FP32 ONNX size: 42.8255672454834
INT8 ONNX size: 10.75895881652832
Compression ratio: 3.980456471279836
Max difference: 0.6589546
Mean difference: 0.1818738
Torch acc: 0.0
FP32 ONNX acc: 0.0
INT8 ONNX acc: 0.0
          Model File size (MB)  Avg latency (ms)  Speedup vs PyTorch
0  PyTorch FP32              —         10.856278            1.000000
1     ONNX FP32      42.825567         12.012787            0.903727
2     ONNX INT8      10.758959         29.074115            0.373400


INT8 quantization reduced the model size significantly from 42.8 MB to 10.7 MB, achieving approximately 4x compression.

However, the measured inference latency for the INT8 model was higher than the FP32 ONNX model. This unexpected result is likely due to dynamic quantization overhead and suboptimal CPU kernel optimization for convolutional models.

Accuracy evaluation was not meaningful in this setup due to the absence of labeled validation data; therefore, results are based on random image inference comparisons.

Overall, ONNX conversion and quantization were successful in terms of deployment readiness and model size reduction, but further optimization (e.g., static quantization or hardware-specific acceleration) would be required for performance gains.